# Lab 01 solution: price a streamed turn and keep a conversation within a budget

The two challenge tasks from `01_model_call.ipynb`, solved and run. This notebook runs on its own: the setup below
repeats the pieces of the lab the tasks use (the client, `price()` from Part C, and Helena's two turns from Part D).

## Setup

The same config cell as every lab, then the client.

In [1]:
from pathlib import Path
import os
import json
from dotenv import dotenv_values
from openai import OpenAI, BadRequestError, omit

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
client = OpenAI()

`price()` is Part C's formula: uncached input, cached input, cache writes and output, each at its price from
`models.env`, per million tokens.

In [3]:
PRICE_INPUT = float(settings["PRICE_INPUT_PER_MILLION"])
PRICE_CACHED_INPUT = float(settings["PRICE_CACHED_INPUT_PER_MILLION"])
PRICE_CACHE_WRITE = float(settings["PRICE_CACHE_WRITE_PER_MILLION"])
PRICE_OUTPUT = float(settings["PRICE_OUTPUT_PER_MILLION"])


def price(usage):
    """Dollars for one call, from its usage and the prices in models.env."""
    details = usage.prompt_tokens_details
    cached = details.cached_tokens or 0
    written = details.cache_write_tokens or 0
    uncached = usage.prompt_tokens - cached - written
    return (
        uncached * PRICE_INPUT + cached * PRICE_CACHED_INPUT + written * PRICE_CACHE_WRITE
        + usage.completion_tokens * PRICE_OUTPUT
    ) / 1_000_000

Part D's conversation: the system message, Helena's first message, the model's reply to it, and her follow-up. As in
the lab, the reply to turn 1 is sent back as an `assistant` message.

In [4]:
SYSTEM = {
    "role": "system",
    "content": "You are a support assistant for the Chinook music shop. Answer in one or two short sentences.",
}
turn_1 = {"role": "user", "content": "Hi, this is Helena. I was charged twice for my order, number 46."}
turn_2 = {"role": "user", "content": "What is my order number?"}

first = client.chat.completions.create(
    model=MODEL, messages=[SYSTEM, turn_1], reasoning_effort=EFFORT or omit, max_completion_tokens=200
)
history = [SYSTEM, turn_1, {"role": "assistant", "content": first.choices[0].message.content}, turn_2]
print(first.choices[0].message.content)

Hi Helena, I’m sorry about the duplicate charge. I can’t access billing records here, so please contact Chinook support with order 46 and the dates and amounts of both charges so they can investigate and arrange a refund if needed.


## Task 1: Price a streamed conversation turn

The stream from Part E, with `include_usage`, sent with `history` instead of the one-question list. The loop keeps the
`chunk.choices` check, because the last chunk, the one with the usage, has an empty `choices` list.

In [5]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=history,
    reasoning_effort=EFFORT or omit,
    max_completion_tokens=200,
    stream=True,
    stream_options={"include_usage": True},
)

pieces = []
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="|")
        pieces.append(chunk.choices[0].delta.content)
    last_chunk = chunk

print()
print("reply:", "".join(pieces))
print("usage:", last_chunk.usage.prompt_tokens, "prompt +", last_chunk.usage.completion_tokens, "completion tokens")
print(f"price: ${price(last_chunk.usage):.7f}")

Your| order| number| is| |46|.|
reply: Your order number is 46.
usage: 112 prompt + 10 completion tokens
price: $0.0000162


The streamed reply names order 46, in 7 pieces. Its prompt was 112 tokens, close to `remembered` in the lab (the turn 1 reply in `history` differs a little from run to run), and the call cost $0.0000162: 112 × 0.10 + 10 × 0.50, per million. Streaming changed when the words arrived, not the bill.

## Task 2: Keep a two-turn conversation within a token budget

Both turns run twice, with two system messages, so the same steps go into a function: turn 1 alone, then turn 2 with
the history, as in Part D. It returns what the budget check needs: the two calls' tokens added up, their two
`finish_reason` values and the second reply.

In [6]:
BUDGET = 180


def two_turns(system_text):
    system = {"role": "system", "content": system_text}
    one = client.chat.completions.create(
        model=MODEL, messages=[system, turn_1], reasoning_effort=EFFORT or omit, max_completion_tokens=200
    )
    history = [system, turn_1, {"role": "assistant", "content": one.choices[0].message.content}, turn_2]
    two = client.chat.completions.create(
        model=MODEL, messages=history, reasoning_effort=EFFORT or omit, max_completion_tokens=200
    )
    total = one.usage.total_tokens + two.usage.total_tokens
    finishes = (one.choices[0].finish_reason, two.choices[0].finish_reason)
    print("turn 1:", one.choices[0].message.content)
    print("turn 2:", two.choices[0].message.content)
    return total, finishes, two.choices[0].message.content

The check reads three things: the total against the budget, both `finish_reason` values (a reply cut off to save
tokens fails, as in Part B), and "46" in the second reply. A pass says the conversation fit and still answered; it does
not say the first reply was helpful.

First Part D's system message, as a baseline:

In [7]:
def check(total, finishes, answer):
    ok = total <= BUDGET and finishes == ("stop", "stop") and "46" in answer
    print(f"{'✅' if ok else '❌'} {total} tokens (budget {BUDGET}), finish_reason {finishes}, names 46: {'46' in answer}")


check(*two_turns(SYSTEM["content"]))

turn 1: Hi Helena, I’m sorry about the duplicate charge. I can’t access billing records here, so please contact Chinook support with order #46 and the dates and amounts of both charges; they can verify it and arrange a refund if needed.
turn 2: Your order number is 46.
❌ 223 tokens (budget 180), finish_reason ('stop', 'stop'), names 46: True


❌ at 223 tokens: both replies finished and the second names 46, but the long first reply, sent twice, puts the pair 43 tokens over the budget.

The first reply goes into the second prompt, so its length counts twice. A system message that asks for shorter
replies cuts both places at once:

In [8]:
check(*two_turns("You are a support assistant for the Chinook music shop. Answer in at most ten words."))

turn 1: Sorry about the duplicate charge, Helena. Please contact support for a refund review.
turn 2: Your order number is 46.
✅ 155 tokens (budget 180), finish_reason ('stop', 'stop'), names 46: True


✅ at 155 tokens: the first reply shrank to one short sentence, both calls ended with `stop`, and turn 2 still names 46, because Helena's own message is in the history even when the reply leaves the number out.

`check(*two_turns(...))` unpacks the three values `two_turns` returns into `check`'s three arguments. The other route,
a smaller `max_completion_tokens` on turn 1 with the old system message, saves tokens only by cutting the first reply
off; the check would then show `length` and fail, which is the Part B lesson again.